# Kaggle ResNet18 SimCLR Runner

Use this notebook on Kaggle GPU for one ResNet18 SimCLR experiment at a time. The training logic stays in the repository scripts. This notebook only prepares Kaggle paths, verifies inputs, runs pretraining, then runs supervised fine-tuning.

## 1. Enable GPU

In Kaggle, open notebook settings and set Accelerator to GPU. Do not use TPU for this pipeline.

In [ ]:
from pathlib import Path
import os
import shutil
import pandas as pd

print('Kaggle input exists:', Path('/kaggle/input').exists())
print('Kaggle working exists:', Path('/kaggle/working').exists())
!nvidia-smi

## 2. Clone or Pull Repository

In [ ]:
REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/kaggle/working/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('REPO_ROOT:', REPO_ROOT)
!git rev-parse --short HEAD

## 3. Install Minimal Dependencies

Kaggle already includes PyTorch. Install only lightweight packages used by the scripts.

In [ ]:
!pip install -q timm scikit-learn matplotlib pandas Pillow

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 4. Edit Kaggle Dataset Paths

After adding your Kaggle Dataset to this notebook, edit these paths to match the folder names under `/kaggle/input`. The helper cell below links them into the repository as `data/processed/...`, so the existing scripts do not need path changes.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path("/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data")

LABELLED_SOURCE = DATA_ROOT / "processed/labelled_4232"
UNLABELLED_SOURCE = DATA_ROOT / "processed/unlabelled_16934"
SYNTHETIC_SOURCE = DATA_ROOT / "processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan"
MANIFEST_SOURCE = DATA_ROOT / "manifests"

OUTPUT_ROOT = Path("/kaggle/working/results/experiments")
PREVIOUS_EXPERIMENT_SOURCE = None

# 'resnet18_covidqu', 'resnet18_imagenet_covidqu', 'resnet18_covidqu_syn', 'resnet18_imagenet_covidqu_syn'
EXPERIMENT_ID = "resnet18_covidqu_syn"

PRETRAIN_EPOCHS = 70
FINETUNE_EPOCHS = None

print("LABELLED_SOURCE:", LABELLED_SOURCE, LABELLED_SOURCE.exists())
print("UNLABELLED_SOURCE:", UNLABELLED_SOURCE, UNLABELLED_SOURCE.exists())
print("SYNTHETIC_SOURCE:", SYNTHETIC_SOURCE, SYNTHETIC_SOURCE.exists())
print("MANIFEST_SOURCE:", MANIFEST_SOURCE, MANIFEST_SOURCE.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

## 5. Link Data and Prepare Manifests

In [ ]:
def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked', target, '->', source)
    else:
        print('WARNING: source missing:', source)

%cd {REPO_ROOT}
replace_path(REPO_ROOT / 'data/processed/labelled_4232', LABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/unlabelled_16934', UNLABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/synthetic_dcgan', SYNTHETIC_SOURCE)

manifest_dir = REPO_ROOT / 'data/manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

for name in ['train.csv', 'val.csv', 'test.csv', 'labelled_all.csv', 'split_summary.json']:
    src = MANIFEST_SOURCE / name
    dst = manifest_dir / name
    if src.exists():
        shutil.copy2(src, dst)
        print('Copied manifest:', dst)
    elif dst.exists():
        print('Using repo manifest:', dst)
    else:
        print('WARNING: missing manifest:', src)

# Normalize synthetic manifest for Kaggle. Stage 1 manifests made on Colab may contain Drive absolute paths.
src_syn_manifest = MANIFEST_SOURCE / 'synthetic_dcgan.csv'
dst_syn_manifest = manifest_dir / 'synthetic_dcgan.csv'
if src_syn_manifest.exists():
    df = pd.read_csv(src_syn_manifest)
    normalized_paths = []
    for _, row in df.iterrows():
        original = Path(str(row['image_path']))
        class_name = row['class_name']
        filename = original.name
        candidates = [
            Path('data/processed/synthetic_dcgan') / class_name / 'images' / filename,
            Path('data/processed/synthetic_dcgan') / class_name / filename,
            Path('data/processed/synthetic_dcgan') / original.name,
        ]
        selected = candidates[0]
        for candidate in candidates:
            if (REPO_ROOT / candidate).exists():
                selected = candidate
                break
        normalized_paths.append(str(selected))
    df['image_path'] = normalized_paths
    df.to_csv(dst_syn_manifest, index=False)
    print('Wrote Kaggle-normalized synthetic manifest:', dst_syn_manifest)
elif dst_syn_manifest.exists():
    print('Using repo synthetic manifest:', dst_syn_manifest)
else:
    rows = []
    class_to_label = {'COVID': 0, 'Lung_Opacity': 1, 'Viral_Pneumonia': 2, 'Normal': 3}
    image_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    if SYNTHETIC_SOURCE.exists():
        for class_name, label in class_to_label.items():
            class_dir = REPO_ROOT / 'data/processed/synthetic_dcgan' / class_name
            search_root = class_dir / 'images' if (class_dir / 'images').exists() else class_dir
            for image_path in sorted(search_root.rglob('*')):
                if image_path.is_file() and image_path.suffix.lower() in image_exts:
                    rows.append({
                        'image_path': str(image_path.relative_to(REPO_ROOT)),
                        'class_name': class_name,
                        'label': label,
                        'source': 'synthetic',
                        'generator': 'dcgan',
                    })
    if rows:
        pd.DataFrame(rows).to_csv(dst_syn_manifest, index=False)
        print('Generated synthetic manifest from Kaggle synthetic folder:', dst_syn_manifest, 'rows=', len(rows))
    else:
        print('WARNING: synthetic_dcgan.csv not found and synthetic images could not be discovered. Synthetic experiments will fail until this is provided.')

!find data -maxdepth 3 -type d | sort | head -40
!ls -lh data/manifests

## 6. Verify Inputs and Scripts

In [ ]:
SYNTHETIC_MANIFEST = REPO_ROOT / 'data/manifests/synthetic_dcgan.csv'

!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_simclr_resnet.py scripts/run_classification_resnet.py
!python scripts/run_simclr_resnet.py --help | grep resume || true

## 7. Resolve Experiment Settings

In [ ]:
EXPERIMENTS = {
    'resnet18_covidqu': {
        'config': 'configs/experiments/resnet18/covidqu.yaml',
        'uses_synthetic': False,
    },
    'resnet18_imagenet_covidqu': {
        'config': 'configs/experiments/resnet18/imagenet_covidqu.yaml',
        'uses_synthetic': False,
    },
    'resnet18_covidqu_syn': {
        'config': 'configs/experiments/resnet18/covidqu_syn.yaml',
        'uses_synthetic': True,
    },
    'resnet18_imagenet_covidqu_syn': {
        'config': 'configs/experiments/resnet18/imagenet_covidqu_syn.yaml',
        'uses_synthetic': True,
    },
}

if EXPERIMENT_ID not in EXPERIMENTS:
    raise ValueError(f'Unsupported EXPERIMENT_ID: {EXPERIMENT_ID}')

EXP = EXPERIMENT_ID
CONFIG = EXPERIMENTS[EXP]['config']
USES_SYNTHETIC = EXPERIMENTS[EXP]['uses_synthetic']
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_simclr_checkpoint.pth'

pretrain_epoch_arg = f'--epochs {PRETRAIN_EPOCHS}' if PRETRAIN_EPOCHS is not None else ''
finetune_epoch_arg = f'--epochs {FINETUNE_EPOCHS}' if FINETUNE_EPOCHS is not None else ''

print('EXP:', EXP)
print('CONFIG:', CONFIG)
print('OUT:', OUT)
print('CKPT:', CKPT)
print('RESUME_CKPT:', RESUME_CKPT)
print('USES_SYNTHETIC:', USES_SYNTHETIC)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)

## 8. Optional: Restore Previous Kaggle Result

If you uploaded a previous result folder as a Kaggle Dataset, this cell copies it into `/kaggle/working` before training so SimCLR can resume from `last_simclr_checkpoint.pth`.

In [ ]:
if PREVIOUS_EXPERIMENT_SOURCE is not None:
    previous = Path(PREVIOUS_EXPERIMENT_SOURCE)
    if not previous.exists():
        raise FileNotFoundError(f'PREVIOUS_EXPERIMENT_SOURCE does not exist: {previous}')
    if OUT.exists():
        shutil.rmtree(OUT)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(previous, OUT)
    print('Restored previous result folder:', previous, '->', OUT)
else:
    print('No previous result folder configured. Starting from existing /kaggle/working output if present, otherwise fresh.')

print('Resume checkpoint exists:', RESUME_CKPT.exists(), RESUME_CKPT)


## 9. Run SimCLR Pretraining

This command resumes automatically from `last_simclr_checkpoint.pth` if it exists.

In [ ]:
if USES_SYNTHETIC:
    !python scripts/run_simclr_resnet.py \
      --config "{CONFIG}" \
      --synthetic-manifest "{SYNTHETIC_MANIFEST}" \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {pretrain_epoch_arg}
else:
    !python scripts/run_simclr_resnet.py \
      --config "{CONFIG}" \
      --real-unlabeled-dir data/processed/unlabelled_16934 \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {pretrain_epoch_arg}

## 10. Run Supervised Fine-Tuning


In [ ]:
!python scripts/run_classification_resnet.py \
  --config "{CONFIG}" \
  --manifest-dir data/manifests \
  --output-dir "{OUT}" \
  --pretrained-checkpoint "{CKPT}" \
  {finetune_epoch_arg}

## 11. Display and Package Results

Download the zip from the Kaggle output panel, or create a Kaggle Dataset from `/kaggle/working/results` if you need to resume later.

In [ ]:
import json

metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([{**{'experiment_id': EXP}, **metrics}]))
else:
    print('WARNING: metrics.json not found:', metrics_path)

!find "{OUT}" -maxdepth 4 -type f | sort
!cd /kaggle/working && zip -qr "{EXP}_results.zip" results/experiments/"{EXP}"
print('Result zip:', Path('/kaggle/working') / f'{EXP}_results.zip')